# Does `aoutools` score a *fully synthetic* dataset exactly as a hand-built control says?

Run this on the **All of Us Researcher Workbench**, on a Hail Genomic Analysis
environment.

This is the **positive-control** tier. Where `validate_scoring_on_aou.ipynb`
*searches the real VDS* for a real variant of each shape — and skips a check when
the window happens not to contain one — this notebook **constructs** a tiny
synthetic VDS and a synthetic weights file in which **every scoring path is
present by design**, computes the expected PRS for every sample *independently of
`aoutools`*, and drives the real public API (`calculate_prs`,
`calculate_prs_batch`) against that control.

Its complement to the other tiers: `tests/integration/` has synthetic data and
hand-derived answers but cannot reach the public API (all three entry points
hard-require a `gs://` output path) and runs against the CI-pinned hail, not the
Workbench's; this notebook closes both gaps and adds a diagnostic toolkit that
*names the likely bug* when a check fails.

## What it covers

Every path pinned in `tests/integration/` and described in `TODO.md`: clean
biallelic SNPs; the effect allele on the **REF** vs the **ALT**, resolved per row
and mixed in one file; **multi-allelic** sites with carriers of different ALTs
(Finding 6); a sample **homozygous for an unnamed ALT** (the maximum `2w` error);
a **no-call** (unknown genotype, must score 0, not `2w`); a **hom-ref** sample
(no entry — reachable only by the row-level offset); a **non-minimally
represented** variant, both multi-allelic (suffix trim) and biallelic (the
`split_multi` passthrough); a variant whose **REF sorts after its ALT**; a
variant **absent from the callset** (must contribute nothing, not even an
offset); a **degenerate** `A/A` weights row; **non-unit and negative** weights;
`log_transform_weight`; chunking; `samples_to_keep`; batch-equals-single; and the
**locus-shift tripwire** that must *raise*.

## The control is the oracle

Expected scores are computed **two independent ways and cross-checked**: (1) pure
Python over the designed genotypes; (2) copy numbers counted from
`hl.vds.to_dense_mt`, which never calls `min_rep`, `split_multi`, or `aoutools`.
If the two disagree, the synthetic VDS was not built as designed. Neither route
runs the code under test — that independence is the whole point.

## If a check fails

A red check is a real finding: a bug in the library, or a false assumption in the
control. **Do not adjust the control to make it pass.** The diagnostics section
decomposes the discrepancy per variant and per sample and names the likely cause.

## Assumptions

- Runs on the Workbench, with a **writable workspace bucket** for the small
  result CSVs (the only cloud I/O; the VDS is built in-memory).
- The synthetic VDS uses the All of Us local-allele (`LGT`/`LA`) encoding with
  reference blocks, so a hom-ref sample has **no entry** — the property the whole
  scoring design hinges on.
- `calculate_pgs` (PGS Catalog download) is **out of scope**: its inputs are not
  synthetic and cannot be checked against a hand-made control. It is covered by
  `validate_public_api_on_aou.ipynb`.


## Setup

**Goal.** Initialise Hail for the Workbench, attach a log handler so `aoutools`'
INFO progress lines are visible, and resolve the workspace bucket for the result
CSVs.

**Details.** `aoutools` logs at INFO but ships only a `NullHandler`, so nothing
prints until a handler is attached; root stays at WARNING to mute Spark/py4j and
only `aoutools` is raised to INFO. No real VDS is read — the only cloud I/O is
the result CSVs written under `OUT`.


In [ ]:
# If aoutools isn't already installed in this environment:
# !pip install -q git+https://github.com/dokyoonkimlab/aoutools.git@dev

import logging
import math
import tempfile
import textwrap
import time
import warnings
from pathlib import Path

import hail as hl
import hailtop.fs as hfs
import pandas as pd

from aoutools import get_workspace_bucket, init_hail
from aoutools.prs import (
    PRSConfig,
    calculate_prs,
    calculate_prs_batch,
    read_prs_weights,
    read_prscs,
)

# Internal seams -- used ONLY by the diagnostics section, where per-variant
# decomposition needs the in-memory scored table. All headline scoring goes
# through the public calculate_prs / calculate_prs_batch above.
from aoutools.prs._calculator import _calculate_prs_chunk, _prepare_mt_split
from aoutools.prs._calculator_utils import _validate_and_prepare_weights_table

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(levelname)-7s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("aoutools").setLevel(logging.INFO)

init_hail()

# The synthetic VDS is built in-memory; the only cloud I/O is the result CSVs.
BUCKET = get_workspace_bucket()
OUT = f"{BUCKET}/aoutools_synthetic_validation/{time.strftime('%Y%m%d-%H%M%S')}"

NOTEBOOK_START = time.perf_counter()
print("output prefix:", OUT)

## 1. Design the synthetic cohort

**Goal.** A single Python description of 10 samples and ~10 variants that is the
source of truth for **both** the VDS and the control. Every scoring shape appears
at least once, with several carriers so each check is *meaningful* (e.g. real
carriers of an unnamed ALT at the multi-allelic REF-effect site — otherwise
Finding 6 could not have failed).

**Assumptions.** Genotypes are given as **global** allele indices (index 0 is the
reference). A sample absent from a variant's dict is homozygous reference there
(and will get a reference block — no entry). `None` is a no-call: an entry that
exists but whose genotype is unknown.

**What a "score" is, in plain terms.** A polygenic score for one person is a
weighted sum: for every variant, take its weight and multiply by how many copies
of the *effect allele* that person carries (0, 1, or 2), then add those products
up. So the whole job is — for each weighted variant, find the person's genotype,
count the right allele, and total it. Each of the three weight files below is
built to stress a different part of that job.

**The three weight files.**

- **`score_main`** — the main exercise. One weighted variant for each shape in the
  table above, deliberately mixed into a single file so every tricky case is
  present at once: the effect allele is sometimes the reference base and sometimes
  the alternate, and the library has to work out which for each row; some rows are
  written "backwards", with the effect and non-effect alleles swapped, to prove
  the order they are written in does not change the result; two rows name a variant
  that is *not* in the VDS at all, and must contribute nothing; one row names the
  same allele twice — a nonsense "degenerate" row that must be ignored; and one
  weight is negative and not a whole number, so the total is real arithmetic, not
  just a count of copies.
- **`score_alt`** — a second, smaller file that re-uses three of the same
  positions with different effect-allele choices and weights. Its only purpose is
  the batch check: computing two different scores in a single pass must give
  exactly the same numbers as computing each one on its own.
- **`score_or`** — a file whose weights are **odds ratios** (a common way effect
  sizes are published) instead of the log-odds a sum assumes. It is scored with
  `log_transform_weight=True`, which tells the library to take the natural log of
  each weight first. It checks that this conversion is done correctly.

**Details.** The `SCORES` dict pairs each file with whether it needs the log
transform, so the control and the library always agree on it.


In [ ]:
SAMPLES = [f"S{i:02d}" for i in range(1, 11)]  # S01..S10

# (locus, alleles, {sample: global_GT or None}, note). Absent sample = hom-ref.
VARIANTS = [
    (
        "chr1:1000",
        ["A", "G"],
        {"S04": [0, 1], "S05": [0, 1], "S06": [1, 1], "S07": [1, 1]},
        (
            "The simplest case: a two-allele SNP whose reference base sorts "
            "before its alternate."
        ),
    ),
    (
        "chr1:2000",
        ["A", "G"],
        {"S04": [0, 1], "S05": [1, 1], "S06": None},
        (
            "A two-allele SNP where one person has a no-call. An unknown "
            "genotype must score nothing, not two reference copies (Trap 4)."
        ),
    ),
    (
        "chr1:3000",
        ["A", "T"],
        {"S04": [0, 1], "S05": [1, 1]},
        (
            "The data carries A/T here, but a weight names A/G. That weighted "
            "variant is not in the data at all, so it must contribute nothing "
            "to anyone, not even a reference offset."
        ),
    ),
    (
        "chr1:4000",
        ["A", "T"],
        {"S04": [0, 1], "S05": [1, 1]},
        (
            "The same missing variant as the row above, but the weight now "
            "labels the other allele as the effect allele. An absent variant "
            "must still score nothing whichever way round it is written."
        ),
    ),
    (
        "chr1:5000",
        ["C", "G", "T"],
        {
            "S02": [0, 1],
            "S03": [0, 1],
            "S04": [0, 2],
            "S05": [0, 2],
            "S06": [1, 2],
            "S07": [1, 1],
        },
        (
            "A multi-allelic site whose people carry different alternates. "
            "After the site is split, a carrier of the alternate a weight did "
            "not name must not be miscounted (Trap 3)."
        ),
    ),
    (
        "chr1:6000",
        ["AGGGC", "A", "GGGGC"],
        {"S04": [0, 2], "S05": [0, 2], "S06": [2, 2], "S07": [0, 1]},
        (
            "A multi-allelic site written non-minimally. Trimming the shared "
            "ending turns AGGGC/GGGGC into a plain A/G SNP at the same "
            "position (Trap 5)."
        ),
    ),
    (
        "chr1:7000",
        ["A", "C", "G"],
        {"S02": [1, 1], "S04": [0, 2], "S05": [2, 2], "S06": [0, 1]},
        (
            "A multi-allelic site where one person is homozygous for the "
            "alternate a weight did not name. This is the worst case for the "
            "reference offset: mishandling it credits two copies the person "
            "does not carry (Trap 3)."
        ),
    ),
    (
        "chr1:8000",
        ["G", "A"],
        {"S04": [0, 1], "S05": [0, 1], "S06": [1, 1], "S07": [1, 1]},
        (
            "A SNP whose reference base sorts after its alternate. "
            "Order-blind matching is what keeps it from being silently "
            "dropped (Traps 1 and 5)."
        ),
    ),
    (
        "chr1:8500",
        ["AAAG", "GAAG"],
        {"S04": [0, 1], "S05": [1, 1]},
        (
            "A two-allele variant written non-minimally. Because it already "
            "has only two alleles, the split step leaves it untouched, so the "
            "library has to reduce it to A/G itself (Trap 5)."
        ),
    ),
    (
        "chr1:9000",
        ["A", "C"],
        {"S04": [0, 1], "S05": [1, 1], "S06": [0, 1]},
        (
            "A plain SNP carrying a negative, non-integer weight, so the "
            "total is real arithmetic rather than a copy count."
        ),
    ),
]

WEIGHTS_SCHEMA = hl.tstruct(
    chr=hl.tstr,
    pos=hl.tint32,
    effect_allele=hl.tstr,
    noneffect_allele=hl.tstr,
    weight=hl.tfloat64,
)


def w(pos, effect, noneffect, weight):
    return {
        "chr": "chr1",
        "pos": pos,
        "effect_allele": effect,
        "noneffect_allele": noneffect,
        "weight": weight,
    }


SCORE_MAIN = [
    w(1000, "G", "A", 1.0),  # ALT-effect
    w(1000, "A", "A", 5.0),  # degenerate: effect == noneffect, must be dropped
    w(2000, "A", "G", 1.0),  # REF-effect; site has a no-call
    w(3000, "A", "G", 1.0),  # unmatched (VDS is A/T); REF-effect -> no offset
    w(4000, "G", "A", 1.0),  # unmatched, mirrored
    w(5000, "C", "T", 1.0),  # REF-effect at a multi-allelic site (Finding 6)
    w(6000, "G", "A", 1.0),  # ALT-effect, suffix-trim normalization
    w(7000, "A", "G", 1.0),  # REF-effect worst case (S02 is C/C)
    w(8000, "A", "G", 1.0),  # ALT-effect at a REF>ALT locus, written swapped
    w(8500, "G", "A", 1.0),  # ALT-effect, biallelic non-minimal
    w(9000, "C", "A", -1.3),  # negative, non-unit weight
]

SCORE_ALT = [
    w(8000, "G", "A", 1.0),  # REF-effect at the REF>ALT locus
    w(1000, "A", "G", 2.0),  # REF-effect, non-unit
    w(5000, "T", "C", 1.0),  # ALT-effect at the multi-allelic site
]

SCORE_OR = [  # weights are odds ratios; log_transform_weight=True
    w(1000, "G", "A", 2.0),
    w(9000, "C", "A", 3.0),
]

# (weight rows, log_transform) per named score. The bool feeds both the control
# and the library config so they always agree on the transform.
SCORES = {
    "score_main": (SCORE_MAIN, False),
    "score_alt": (SCORE_ALT, False),
    "score_or": (SCORE_OR, True),
}

print(f"{len(SAMPLES)} samples, {len(VARIANTS)} synthetic VDS variants\n")
for ls, al, _c, note in VARIANTS:
    body = textwrap.fill(
        note, width=72, initial_indent="    ", subsequent_indent="    "
    )
    print(f"{ls}  {'/'.join(al)}\n{body}\n")

## 2. Build the synthetic VDS

**Goal.** A real-hail `VariantDataset` with the All of Us local-allele encoding,
built from the design above.

**Assumptions & details.** Built via
`hl.vds.VariantDataset.from_merged_representation` (the same technique as
`tests/integration/conftest.py`) rather than by assembling `reference_data` /
`variant_data` by hand, because it reproduces the one property scoring depends
on: a homozygous-reference sample is a **reference block**, not a variant entry,
so hail filters it out of the entry stream and no per-entry default can reach it.
The check at the end proves that sparsity holds. The locus-shifting variant is
built as a *separate* VDS — `split_multi` runs over the whole VDS, so a single
such row would make every other cell raise.


In [ ]:
def build_vds(variants):
    """Build a synthetic VDS from a (locus, alleles, carriers, note) list.

    Ported from tests/integration/conftest.py::_build_vds. Absent samples get a
    reference block (no entry); a None genotype is a no-call.
    """
    schema = hl.tstruct(
        locus_str=hl.tstr,
        alleles=hl.tarray(hl.tstr),
        s=hl.tstr,
        gt=hl.tarray(hl.tint32),
        la=hl.tarray(hl.tint32),
        GQ=hl.tint32,
        END=hl.tint32,
    )
    entries = []
    for locus_str, alleles, carriers, _note in variants:
        pos = int(locus_str.split(":")[1])
        for sample in SAMPLES:
            if sample not in carriers:
                # Homozygous reference: a reference block, not a variant entry.
                entries.append(
                    {
                        "locus_str": locus_str,
                        "alleles": [alleles[0]],
                        "s": sample,
                        "gt": [0, 0],
                        "la": [0],
                        "GQ": 99,
                        "END": pos,
                    }
                )
                continue
            global_gt = carriers[sample]
            local_alleles = (
                [0] if global_gt is None else sorted({0, *global_gt})
            )
            local_gt = (
                None
                if global_gt is None
                else [local_alleles.index(i) for i in global_gt]
            )
            entries.append(
                {
                    "locus_str": locus_str,
                    "alleles": alleles,
                    "s": sample,
                    "gt": local_gt,
                    "la": local_alleles,
                    "GQ": 40,
                    "END": None,
                }
            )
    ht = hl.Table.parallelize(entries, schema)
    ht = ht.annotate(
        locus=hl.parse_locus(ht.locus_str, reference_genome="GRCh38")
    )
    ht = ht.annotate(
        LGT=hl.or_missing(hl.is_defined(ht.gt), hl.call(ht.gt[0], ht.gt[1])),
        LA=ht.la,
    )
    ht = ht.drop("gt", "la", "locus_str")
    mt = ht.to_matrix_table(row_key=["locus", "alleles"], col_key=["s"])
    vds = hl.vds.VariantDataset.from_merged_representation(mt, is_split=False)
    vds.validate()  # fail loudly on a malformed VDS
    return vds


vds = build_vds(VARIANTS)
vds = vds.checkpoint(hl.utils.new_temp_file(extension="vds"), overwrite=True)
K = vds.variant_data.count_rows()
print(f"built VDS: {K} variants, {len(SAMPLES)} samples")

# The locus-shifting tripwire lives in its own VDS (see the tripwire section).
LOCUS_SHIFT = [("chr1:1001", ["GG", "G", "GT"], {"S02": [0, 1]}, "tripwire")]
vds_shift = build_vds(LOCUS_SHIFT)

# The property everything rests on: a hom-ref sample has NO entry.
per = vds.variant_data.select_cols(n=hl.agg.count()).cols().to_pandas()
ENTRY_COUNT = dict(zip(per["s"], per["n"].astype(int), strict=True))
ZERO_ENTRY = {s for s, n in ENTRY_COUNT.items() if n == 0}
print(f"\nentries per sample in variant_data: {sorted(ENTRY_COUNT.values())}")
print(f"samples with ZERO entries (hom-ref everywhere): {sorted(ZERO_ENTRY)}")
assert ZERO_ENTRY, (
    "expected some sample to be hom-ref at every variant and therefore absent "
    "from the entry stream -- without one, the hom-ref offset is untested"
)
print("OK: hom-ref samples are invisible to the entry stream.")

## 3. The positive control (and an independent cross-check)

**Goal.** For each score, the expected PRS per sample, computed **without touching
`aoutools`**.

**Details.** `control_score` reads copies straight off the designed genotypes:
for an **ALT**-effect weight, the count of that ALT's global index (hom-ref and
no-call → 0); for a **REF**-effect weight, `2 − n_non_ref` for a called genotype,
**2** for a hom-ref (no-entry) sample, **0** for a no-call; an unmatched weights
row contributes 0 to everyone and is excluded from `n_matched`; a degenerate
`A/A` row matches nothing. `match_variant` maps a weights row to the VDS ALT it
names using the same minimal-representation **set** compare the library uses.

**Cross-check.** `oracle_score` recomputes the same expectation by counting the
chosen allele index in `hl.vds.to_dense_mt` — reading genotypes back out of the
*actual* VDS, with no `min_rep`, no `split_multi`, no `aoutools`. Asserting the
two agree also proves `from_merged_representation` encoded the design as intended.


In [ ]:
def minimal_representation(ref, alt):
    """Name a variant the way a GWAS would. Pure Python, no hail.

    Trim the shared suffix, then the shared prefix, keeping >=1 base of each.
    Returns (ref, alt, shift); shift > 0 only if a shared prefix was trimmed
    (which MOVES the locus -- the case the library refuses to key across).
    """
    while len(ref) > 1 and len(alt) > 1 and ref[-1] == alt[-1]:
        ref, alt = ref[:-1], alt[:-1]
    shift = 0
    while len(ref) > 1 and len(alt) > 1 and ref[0] == alt[0]:
        ref, alt = ref[1:], alt[1:]
        shift += 1
    return ref, alt, shift


def match_variant(effect, noneffect, alleles):
    """Which ALT index (1-based into `alleles`) does this pair name? Or None.

    Mirrors the library's unordered-set match against each ALT's minimal form.
    A degenerate pair (effect == noneffect) names nothing.
    """
    if effect == noneffect:
        return None
    for i in range(1, len(alleles)):
        mref, malt, shift = minimal_representation(alleles[0], alleles[i])
        if shift == 0 and {mref, malt} == {effect, noneffect}:
            return i, mref, malt
    return None


def _carriers_of(pos):
    for ls, alleles, carriers, _n in VARIANTS:
        if ls == f"chr1:{pos}":
            return alleles, carriers
    return None, None


def control_score(rows, log_transform=False):
    """Expected {sample: prs} and n_matched, computed from the design alone."""
    prs = {s: 0.0 for s in SAMPLES}
    matched = set()
    for r in rows:
        alleles, carriers = _carriers_of(r["pos"])
        if alleles is None:
            continue  # weights-only 'ghost' locus: no VDS variant here
        m = match_variant(r["effect_allele"], r["noneffect_allele"], alleles)
        if m is None:
            continue  # absent variant or degenerate row: contributes nothing
        i, mref, _malt = m
        ref_is_effect = r["effect_allele"] == mref
        weight = math.log(r["weight"]) if log_transform else r["weight"]
        matched.add((r["pos"], i))
        for s in SAMPLES:
            if s not in carriers:  # hom-ref, no entry
                copies = 2.0 if ref_is_effect else 0.0
            elif carriers[s] is None:  # no-call: unknown genotype
                copies = 0.0
            else:
                gt = carriers[s]
                copies = (
                    float(sum(g == 0 for g in gt))
                    if ref_is_effect
                    else float(sum(g == i for g in gt))
                )
            prs[s] += weight * copies
    return prs, len(matched)


def oracle_score(a_vds, rows, log_transform=False):
    """Independent expectation: copy counts from the dense matrix."""
    dense = hl.vds.to_dense_mt(a_vds)
    dense = dense.annotate_entries(GT=hl.vds.lgt_to_gt(dense.LGT, dense.LA))
    dense = dense.checkpoint(hl.utils.new_temp_file(), overwrite=True)
    prs = {s: 0.0 for s in SAMPLES}
    for r in rows:
        alleles, _carriers = _carriers_of(r["pos"])
        if alleles is None:
            continue
        m = match_variant(r["effect_allele"], r["noneffect_allele"], alleles)
        if m is None:
            continue
        i, mref, _malt = m
        effect_index = 0 if r["effect_allele"] == mref else i
        weight = math.log(r["weight"]) if log_transform else r["weight"]
        key = f"chr1:{r['pos']}|{','.join(alleles)}"
        row_key = hl.str(dense.locus) + "|" + hl.delimit(dense.alleles, ",")
        idx = hl.literal({key: effect_index}).get(row_key, -1)
        copies = hl.if_else(
            hl.is_defined(dense.GT) & (idx >= 0),
            hl.if_else(dense.GT[0] == idx, 1, 0)
            + hl.if_else(dense.GT[1] == idx, 1, 0),
            0,
        )
        df = dense.select_cols(n=hl.agg.sum(copies)).cols().to_pandas()
        for s, n in zip(df["s"], df["n"], strict=True):
            prs[s] += weight * float(n)
    return prs


# Self-test the pure-Python helpers before trusting them (they are the oracle).
assert minimal_representation("AGGGC", "GGGGC") == ("A", "G", 0)  # suffix trim
assert minimal_representation("GG", "GT") == ("G", "T", 1)  # locus MOVES
assert minimal_representation("AAAG", "GAAG") == ("A", "G", 0)
assert match_variant("A", "A", ["A", "G"]) is None  # degenerate
assert match_variant("G", "A", ["A", "G"]) == (1, "A", "G")  # unordered
assert match_variant("A", "G", ["A", "T"]) is None  # absent
print("control-helper self-test OK\n")

CONTROL = pd.DataFrame(index=SAMPLES)
CONTROL.index.name = "person_id"
CONTROL_N = {}
for name, (rows, logt) in SCORES.items():
    prs, n = control_score(rows, log_transform=logt)
    CONTROL[name] = [prs[s] for s in SAMPLES]
    CONTROL_N[name] = n
    # Cross-check the control against the dense-matrix oracle.
    orac = oracle_score(vds, rows, log_transform=logt)
    bad = {s for s in SAMPLES if abs(orac[s] - prs[s]) > 1e-9}
    assert not bad, (
        f"control and dense-matrix oracle disagree for {name} at {sorted(bad)} "
        "-- the synthetic VDS was not built as designed, or the control "
        "arithmetic is wrong. Neither runs aoutools; this is a real finding."
    )

print("control == dense-matrix oracle for every score  OK\n")
print("CONTROL (expected PRS per sample):")
print(CONTROL.round(4).to_string())
print("\nexpected n_matched:", CONTROL_N)

## 4. Verification & diagnostics harness

**Goal.** One `verify` used by every scoring section, and a `diagnose` toolkit
that captures *what went wrong* and runs only when something disagrees.

**Details.** `verify` compares the library's `{sample: prs}` to the control at
tolerance `1e-9`, optionally checks `n_matched`, and records the outcome in
`RESULTS`. `diagnose` does three things: (1) **decomposes** the score one variant
at a time (via the in-memory seam) to localise *which shape* diverges; (2)
**dumps** the split-MatrixTable row annotations (`canonical_alleles`,
`weight_per_alt_copy`, `hom_ref_offset`, `ref_is_effect`) so you can see whether
the join matched and how orientation resolved; (3) **classifies** the failure
signature and points at the matching finding in `TODO.md`.


In [ ]:
RESULTS = []
SKIPPED = []


def raw_table(rows):
    return hl.Table.parallelize(rows, WEIGHTS_SCHEMA)


def print_scores(got, expected=None, label="calculate_prs"):
    """Print the per-sample scores the API returned, next to the control."""
    cols = {}
    if expected is not None:
        cols["control"] = pd.Series(dict(expected), dtype=float)
    cols[label] = pd.Series(got, dtype=float)
    tab = pd.DataFrame(cols)
    if expected is not None:
        tab["delta"] = tab[label] - tab["control"]
    tab.index.name = "person_id"
    print(tab.round(4).to_string())


def verify(name, got, expected, n_matched=None, exp_n_matched=None):
    """Compare {sample: prs} to the control; record pass/fail in RESULTS."""
    mism = {
        s: (float(expected[s]), float(got[s]))
        for s in SAMPLES
        if s in got and abs(float(expected[s]) - float(got[s])) > 1e-9
    }
    n_ok = n_matched is None or n_matched == exp_n_matched
    ok = not mism and n_ok
    RESULTS.append((name, ok, len(mism)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    if n_matched is not None:
        flag = "" if n_ok else "   <-- MISMATCH"
        print(
            f"      n_matched: got {n_matched}, expected {exp_n_matched}{flag}"
        )
    for s, (exp, act) in list(mism.items())[:8]:
        print(
            f"      {s}: control {exp:+.4f}, library {act:+.4f}  "
            f"(delta {act - exp:+.4f})"
        )
    return ok


def diagnose(name, rows, got, expected, log_transform=False):
    """Localise and classify a discrepancy. Called only when verify fails."""
    print("\n" + "=" * 70)
    print(f"DIAGNOSTICS for {name}")
    print("=" * 70)
    cfg = PRSConfig(include_n_matched=True, log_transform_weight=log_transform)

    print(
        "\n(1) Per-variant decomposition (control vs library, one row at a "
        "time):"
    )
    for r in rows:
        ctrl_v, _ = control_score([r], log_transform=log_transform)
        weights = _validate_and_prepare_weights_table(raw_table([r]), cfg)
        df = _calculate_prs_chunk(weights, vds, cfg).to_pandas()
        lib_v = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
        diff = [s for s in SAMPLES if abs(ctrl_v[s] - lib_v[s]) > 1e-9]
        tag = "   <-- DISAGREES" if diff else ""
        print(
            f"    chr1:{r['pos']} {r['effect_allele']}/"
            f"{r['noneffect_allele']} w={r['weight']}: "
            f"{len(diff)} sample(s) differ{tag}"
        )
        for s in diff[:6]:
            print(
                f"        {s}: control {ctrl_v[s]:+.3f}, "
                f"library {lib_v[s]:+.3f}"
            )

    print(
        "\n(2) Split MatrixTable rows (did the join match, how did "
        "orientation resolve):"
    )
    weights = _validate_and_prepare_weights_table(raw_table(rows), cfg)
    mt = _prepare_mt_split(vds=vds, weights_table=weights, config=cfg)
    rdf = (
        mt.rows()
        .select(
            "canonical_alleles",
            "weight_per_alt_copy",
            "hom_ref_offset",
            "ref_is_effect",
        )
        .to_pandas()
    )
    print(rdf.to_string(index=False))

    print("\n(3) Failure signature:")
    deltas = {
        s: float(got[s]) - float(expected[s])
        for s in SAMPLES
        if abs(float(got[s]) - float(expected[s])) > 1e-9
    }
    if not deltas:
        print(
            "    No per-sample mismatch. If n_matched was short, a whole "
            "variant was dropped -- see (1) for which, and suspect the join "
            "(allele order / min_rep). TODO.md Findings 4 and 5."
        )
        return
    off = set(deltas)
    values = {round(v, 9) for v in deltas.values()}
    if off == set(SAMPLES) and len(values) == 1:
        print(
            f"    Constant shift of {values.pop():+.4f} across ALL samples. "
            "Ranking-preserving. Suspect the 2w offset or a scale factor. "
            "TODO.md: the old ref_is_effect_allele=True shape."
        )
    elif off <= ZERO_ENTRY:
        print(
            "    Only hom-ref (no-entry) samples are wrong. The row-level "
            "offset is not reaching absent samples. TODO.md Task 2."
        )
    else:
        print(
            "    Per-sample, GENOTYPE-DEPENDENT error -- this reorders the "
            "cohort. Suspect downcoding / n_non_ref / a zeroed hom-ref. "
            "TODO.md Findings 6 and 1."
        )
    for s in sorted(deltas)[:8]:
        print(
            f"        {s}: delta {deltas[s]:+.4f}  (entries={ENTRY_COUNT[s]})"
        )

## 5. Score through the public API — single (`calculate_prs`)

**Goal.** Drive the real user-facing entry point — the one no offline test can
reach, because it hard-requires a `gs://` output path — write the result CSV to
the bucket, read it back, and compare to the control.

**Details.** Four runs against `score_main`: the default config (with
`include_n_matched`, so the matched-variant count is checked too); `chunk_size=1`
to prove the chunk boundaries **partition** the weights (an overlap double-counts,
a gap drops — either changes the total); `samples_to_keep` for the sample-filter
path; and `score_or` with `log_transform_weight=True`. Each is verified against
`CONTROL`, and `diagnose` is invoked automatically on any failure.


In [ ]:
raw_main = raw_table(SCORE_MAIN)

# 5a. Default config.
path = f"{OUT}/main.csv"
calculate_prs(
    weights_table=raw_main,
    vds=vds,
    output_path=path,
    config=PRSConfig(include_n_matched=True),
)
with hfs.open(path, "r") as fh:
    df = pd.read_csv(fh)
got = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
n = int(df["n_matched"].iloc[0])

print("Per-sample scores from calculate_prs, next to the control:")
print_scores(got, CONTROL["score_main"])
if not verify(
    "5a. calculate_prs, default",
    got,
    CONTROL["score_main"],
    n,
    CONTROL_N["score_main"],
):
    diagnose("score_main", SCORE_MAIN, got, CONTROL["score_main"])

In [ ]:
# 5b. chunk_size=1 -- must equal the single-chunk result and the control.
path = f"{OUT}/main_chunk1.csv"
calculate_prs(
    weights_table=raw_main,
    vds=vds,
    output_path=path,
    config=PRSConfig(chunk_size=1),
)
with hfs.open(path, "r") as fh:
    df = pd.read_csv(fh)
got = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
print_scores(got, CONTROL["score_main"])
if not verify(
    "5b. calculate_prs, chunk_size=1 (partitioning)", got, CONTROL["score_main"]
):
    diagnose(
        "score_main (chunk_size=1)", SCORE_MAIN, got, CONTROL["score_main"]
    )

In [ ]:
# 5c. samples_to_keep -- a subset, still scored against the control.
keep = ["S01", "S04", "S05", "S06", "S07"]
path = f"{OUT}/main_subset.csv"
calculate_prs(
    weights_table=raw_main,
    vds=vds,
    output_path=path,
    config=PRSConfig(samples_to_keep=keep),
)
with hfs.open(path, "r") as fh:
    df = pd.read_csv(fh)
got = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
print_scores(got, CONTROL["score_main"].loc[keep])
sub_ok = set(got) == set(keep)
if not sub_ok:
    print(
        f"FAIL  5c. samples_to_keep returned {sorted(got)}, expected "
        f"{sorted(keep)}"
    )
    RESULTS.append(("5c. calculate_prs, samples_to_keep", False, 0))
else:
    verify(
        "5c. calculate_prs, samples_to_keep",
        got,
        CONTROL["score_main"].loc[keep],
    )

In [ ]:
# 5d. log_transform_weight -- weights given as odds ratios.
raw_or = raw_table(SCORE_OR)
path = f"{OUT}/or.csv"
calculate_prs(
    weights_table=raw_or,
    vds=vds,
    output_path=path,
    config=PRSConfig(log_transform_weight=True),
)
with hfs.open(path, "r") as fh:
    df = pd.read_csv(fh)
got = dict(zip(df["person_id"], df["prs"].astype(float), strict=True))
print("log_transform takes ln(weight): odds ratios 2.0, 3.0 -> 0.6931, 1.0986")
print_scores(got, CONTROL["score_or"])
if not verify(
    "5d. calculate_prs, log_transform_weight", got, CONTROL["score_or"]
):
    diagnose("score_or", SCORE_OR, got, CONTROL["score_or"], log_transform=True)

## 6. Score through the public API — batch (`calculate_prs_batch`)

**Goal.** Score several weight files in one pass; confirm each column matches the
control, that batch equals single for the shared score, and that a
log-transformed score can be batched too.

**Details.** The batch calculator never filters rows — it masks with
`hl.if_else` where the single-score path uses `filter_rows` — so the two use
different mechanisms and must still produce identical numbers, including for the
row-level offset, which batch has to gate on `is_valid_{score}` explicitly.

**Why `score_or` is a separate batch call.** `log_transform_weight` is a setting
on the *whole* `PRSConfig`, not per score, and one `calculate_prs_batch` call
applies one config to every file in it. Putting `score_or` (odds ratios, which
need the log) in the same call as `score_main` / `score_alt` (already on the right
scale) would log-transform *those* too and score them wrongly. So `score_main` and
`score_alt` share the first call, and `score_or` gets its own call with
`log_transform_weight=True`. Both are verified against the control below.


In [ ]:
raw_alt = raw_table(SCORE_ALT)
path = f"{OUT}/batch.csv"
calculate_prs_batch(
    weights_tables_map={"score_main": raw_main, "score_alt": raw_alt},
    vds=vds,
    output_path=path,
    config=PRSConfig(include_n_matched=True),
)
with hfs.open(path, "r") as fh:
    bdf = pd.read_csv(fh)
print(bdf.round(4).to_string(index=False), "\n")

for score in ("score_main", "score_alt"):
    got = dict(zip(bdf["person_id"], bdf[score].astype(float), strict=True))
    n = int(bdf[f"n_matched_{score}"].iloc[0])
    if not verify(
        f"6. calculate_prs_batch, {score}",
        got,
        CONTROL[score],
        n,
        CONTROL_N[score],
    ):
        diagnose(score, SCORES[score][0], got, CONTROL[score])

# Batch must equal single for the shared score, exactly.
batch_main = dict(
    zip(bdf["person_id"], bdf["score_main"].astype(float), strict=True)
)
delta = max(abs(batch_main[s] - CONTROL["score_main"][s]) for s in SAMPLES)
ok = delta < 1e-9
RESULTS.append(("6. batch == single (score_main)", ok, 0))
print(
    f"{'PASS' if ok else 'FAIL'}  6. batch == single (largest delta "
    f"{delta:.2g})"
)

# score_or must be batched in its OWN call: log_transform_weight is a
# whole-config setting, so including it in the batch above would wrongly
# log-transform score_main and score_alt too. It is verified through the batch
# path in its own call here.
or_path = f"{OUT}/batch_or.csv"
calculate_prs_batch(
    weights_tables_map={"score_or": raw_table(SCORE_OR)},
    vds=vds,
    output_path=or_path,
    config=PRSConfig(include_n_matched=True, log_transform_weight=True),
)
with hfs.open(or_path, "r") as fh:
    odf = pd.read_csv(fh)
got = dict(zip(odf["person_id"], odf["score_or"].astype(float), strict=True))
n = int(odf["n_matched_score_or"].iloc[0])
print_scores(got, CONTROL["score_or"], label="calculate_prs_batch")
if not verify(
    "6. calculate_prs_batch, score_or (log_transform)",
    got,
    CONTROL["score_or"],
    n,
    CONTROL_N["score_or"],
):
    diagnose("score_or", SCORE_OR, got, CONTROL["score_or"], log_transform=True)

## 7. The reader and the local→GCS staging path

**Goal.** Exercise `read_prs_weights`, including `_stage_local_file_to_gcs` (a
local file copied into the workspace bucket so Hail's Spark cluster can read
it) — a step no offline test reaches — and the validation filters.

**Details.** A clean file must parse and have its chromosome column standardized
to `chr…`; a file mixing a non-ACGT allele, a zero weight, and a null-coordinate
row must have exactly those rows dropped (not rejected wholesale); a genuine
duplicate must raise. `read_prscs` is now **deprecated** — a thin wrapper for the
fixed header-less PRS-CS layout — so 7d confirms it emits a deprecation warning
and still parses by delegating to `read_prs_weights`.


In [ ]:
tmp = Path(tempfile.mkdtemp())
HEADER = "chr\tpos\teffect_allele\tother_allele\tweight"
CMAP = {
    "chr": "chr",
    "pos": "pos",
    "effect_allele": "effect_allele",
    "noneffect_allele": "other_allele",
    "weight": "weight",
}


def write_tsv(name, lines):
    p = tmp / name
    p.write_text("\n".join([HEADER, *lines]) + "\n")
    return str(p)


# 7a. Clean file: two rows, mixed chr prefix -> both standardized to chr...
good = write_tsv("good.tsv", ["1\t1000\tA\tG\t0.1", "chr2\t2000\tC\tT\t0.2"])
ht = read_prs_weights(
    file_path=good,
    header=True,
    column_map=CMAP,
    delimiter="\t",
    validate_alleles=True,
    missing="",
    force=True,
)
ok = ht.count() == 2 and sorted(ht.chr.collect()) == ["chr1", "chr2"]
RESULTS.append(("7a. read_prs_weights parses + standardizes chr", ok, 0))
print(
    f"{'PASS' if ok else 'FAIL'}  7a. read_prs_weights: {ht.count()} rows, "
    f"chrs {sorted(ht.chr.collect())}"
)

# 7b. Bad rows dropped (not the whole file): non-ACGT, zero weight, null coords.
bad = write_tsv(
    "bad.tsv",
    [
        "1\t1000\tA\tG\t0.1",  # keep
        "1\t2000\tN\tT\t0.2",  # non-ACGT -> dropped by validate_alleles
        "1\t3000\tC\tT\t0",  # zero weight -> dropped
        "\t\tA\tG\t0.4",  # null coordinates -> dropped
    ],
)
ht = read_prs_weights(
    file_path=bad,
    header=True,
    column_map=CMAP,
    delimiter="\t",
    validate_alleles=True,
    missing="",
    force=True,
)
ok = ht.count() == 1 and ht.pos.collect() == [1000]
RESULTS.append(("7b. read_prs_weights drops bad rows, keeps the rest", ok, 0))
print(
    f"{'PASS' if ok else 'FAIL'}  7b. bad rows dropped, kept {ht.pos.collect()}"
)

# 7c. A genuine duplicate must raise.
dup = write_tsv("dup.tsv", ["1\t1000\tA\tG\t0.1", "1\t1000\tA\tG\t0.2"])
try:
    read_prs_weights(
        file_path=dup,
        header=True,
        column_map=CMAP,
        delimiter="\t",
        validate_alleles=True,
        missing="",
        force=True,
    )
except ValueError as e:
    ok = "uplicate" in str(e)
    print(
        f"{'PASS' if ok else 'FAIL'}  7c. duplicate raised: "
        f"{str(e).splitlines()[0][:70]}"
    )
else:
    ok = False
    print("FAIL  7c. a genuine duplicate did NOT raise")
RESULTS.append(("7c. read_prs_weights rejects a true duplicate", ok, 0))

# 7d. read_prscs is deprecated: it must warn, but still parse the header-less
#     chr, id, pos, A1(effect), A2(noneffect), weight layout by delegating to
#     read_prs_weights.
prscs = tmp / "prscs.tsv"
prscs.write_text("1\trs1\t1000\tA\tG\t0.1\n1\trs2\t2000\tC\tT\t0.2\n")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ht = read_prscs(str(prscs))
warned = any(issubclass(c.category, DeprecationWarning) for c in caught)
ok = ht.count() == 2 and warned
RESULTS.append(("7d. read_prscs warns and still parses the layout", ok, 0))
print(
    f"{'PASS' if ok else 'FAIL'}  7d. read_prscs: {ht.count()} rows, "
    f"deprecation warned={warned}"
)

## 8. The locus-shift tripwire (must raise)

**Goal.** Confirm a variant whose locus *moves* under `min_rep` cannot be scored
silently: when such a variant reaches `split_multi`, the run fails **loudly**.

**Details.** `chr1:1001 [GG, G, GT]` minreps its SNP allele to `G/T` at
chr1:**1002**. `hl.vds.split_multi` will not relocate a row, so it raises
(`filter_changed_loci=False`, the default). No variant of this shape exists in
All of Us (0 of 6,001,424 ALTs measured), so this is a **tripwire** against a
future change in variant representation — not a crash risk. Do **not** silence it
with `filter_changed_loci=True`.

**Where the tripwire lives — a subtlety this notebook turned up.** The raise
happens *inside* `split_multi`, so it fires only for a variant that reaches the
split step. That is what `tests/integration/` exercises through the internal
`_calculate_prs_chunk` seam, and what the gated check below does. The **public**
`calculate_prs` behaves differently: it builds its per-chunk interval prefilter
at the *weights* locus (chr1:1002) and runs `hl.vds.filter_intervals` on the
still-multi-allelic VDS, whose row is keyed at chr1:1001 — so the variant is
**dropped before `split_multi` ever sees it** and is scored `0` with `n_matched`
`0`, a *silent drop* rather than a raise. That is the documented **Finding 5**
behavior (`TODO.md`) — real in principle, but moot in AoU, where no locus shifts.
The observation cell records that public-path drop so the two layers (the loud
split-step tripwire, and the quiet prefilter drop in front of it) are not
confused. It means the "fail loudly" guarantee holds at the split step, not for
a downstream-named variant the prefilter removes first.


In [ ]:
# The tripwire fires inside split_multi, so only a variant that survives to the
# split step reaches it. Exercise it the way tests/integration/ does -- the
# internal _calculate_prs_chunk seam, which splits the whole (unfiltered) VDS.
# (Through the public calculate_prs the variant is removed first by the
# per-chunk interval prefilter; the observation below shows that.)
weights_shift = _validate_and_prepare_weights_table(
    raw_table([w(1002, "T", "G", 1.0)]), PRSConfig()
)
try:
    _calculate_prs_chunk(weights_shift, vds_shift, PRSConfig()).to_pandas()
except Exception as e:  # noqa: BLE001 -- any hail error that names the shift
    msg = str(e)
    ok = "non-left-aligned" in msg or "min_rep moved" in msg
    print(
        f"{'PASS' if ok else 'FAIL'}  8. locus-shift raises at the split "
        f"step: {msg.splitlines()[0][:60]}"
    )
else:
    ok = False
    print("FAIL  8. a locus-shifting variant was split without raising")
RESULTS.append(("8. locus-shift tripwire raises (split step)", ok, 0))

# Observation (not a gate): through the PUBLIC API the interval prefilter is
# built at the weights locus chr1:1002 while the variant's row is at chr1:1001,
# so filter_intervals drops it before split_multi -- Finding 5's silent drop,
# moot in AoU. Shown so it is not mistaken for the tripwire above. Wrapped in
# try/except so that if a future release makes the prefilter reach the variant,
# the resulting raise is reported rather than crashing the cell.
try:
    calculate_prs(
        weights_table=raw_table([w(1002, "T", "G", 1.0)]),
        vds=vds_shift,
        output_path=f"{OUT}/shift.csv",
        config=PRSConfig(include_n_matched=True),
    )
    with hfs.open(f"{OUT}/shift.csv", "r") as fh:
        sdf = pd.read_csv(fh)
    print(
        f"      (note) via calculate_prs the variant is prefiltered out: "
        f"n_matched={int(sdf['n_matched'].iloc[0])}, distinct scores="
        f"{sorted(set(sdf['prs'].astype(float)))} -- Finding 5 silent drop"
    )
except Exception as e:  # noqa: BLE001
    print(
        f"      (note) via calculate_prs the variant now reaches split_multi "
        f"and raises: {str(e).splitlines()[0][:55]}"
    )

## Summary

Every check must pass. A failure is a real finding — see the diagnostics printed
above the failing check. The final cell raises if any check failed; anything
skipped is listed separately so it is never mistaken for a pass.


In [ ]:
summary = pd.DataFrame(RESULTS, columns=["check", "passed", "n_mismatched"])
print(summary.to_string(index=False))

n_failed = int((~summary["passed"]).sum())
print()
if SKIPPED:
    print("SKIPPED -- NOT validated:")
    for s in SKIPPED:
        print(f"  {s}")
    print()

if n_failed:
    raise AssertionError(
        f"{n_failed} check(s) FAILED against the synthetic control. This is a "
        "real finding: a bug in aoutools, or a false assumption in the "
        "control. Do NOT adjust the control to make it pass -- it is computed "
        "independently of the code under test. See the diagnostics above."
    )
print(f"All {len(summary)} checks passed against the synthetic control.")

## Runtime


In [ ]:
elapsed = time.perf_counter() - NOTEBOOK_START
print(f"Whole-notebook wall time: {elapsed:.1f} s  ({elapsed / 60:.1f} min)")
print(
    "Measured from Setup to here; meaningful only on a clean top-to-bottom "
    "run. The data is tiny, so most of this is fixed Spark/VDS overhead."
)